In [1]:
import trainer2 as trainer

/home/gowrav8849/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import torch
from torch_geometric.datasets import KarateClub
# Load the dataset
dataset = KarateClub()
# Access the graph
data = dataset[0]
print(data)
# Example output: Data(x=[34, 34], edge_index=[2, 78], y=[34])


In [15]:
from torch.utils.data import DataLoader
from train_functions import build_A_batch
import torch.nn.functional as F
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def load_data(dataset_name, args):
    # Data Loading ----------------------------------
    dataset = trainer.load_nc_dataset(dataset_name, args.sub_dataname)
    data_ = trainer.nc_to_data(dataset, dataset_name, device)
    labels = data_.y
    unique_labels = torch.unique(labels)
    label_map = {old.item(): i for i, old in enumerate(unique_labels)}
    data_.y = torch.tensor([label_map[x.item()] for x in labels], device=labels.device)
    num_classes = int(data_.y.max().item()) + 1

    # Normalize features
    data_.x = data_.x / data_.x.sum(dim=1, keepdim=True).clamp(min=1)
    num_nodes = data_.num_nodes


    return data_

def get_emb(model, data, batching=True, batch_size=1024, return_logits=True):
    model.eval()
    device = data.x.device
    num_nodes = data.num_nodes
    nodes = torch.arange(num_nodes, device=device)

    if not batching:
        with torch.no_grad():
            out = model(data.x, data.edge_index)
        return out
    # ---- batching ----
    loader = DataLoader(nodes, batch_size=batch_size, shuffle=False)
    out_list = []
   
    with torch.no_grad(): 
        for batch_nodes in loader:
            batch_nodes = batch_nodes.to(device)

            A_batch = build_A_batch(
                data.edge_index,
                batch_nodes,
                data.num_nodes,
                device
            )
            out_batch = model(data.x, data.edge_index, A_batch, batch_nodes)

            out_list.append(out_batch)

    # concatenate all batches
    out = torch.cat(out_list, dim=0)

    return out

In [3]:
import argparse
import sys
def get_args():
    parser = argparse.ArgumentParser(description="LINKX Training Config")

    parser.add_argument('--model_name', type=str, default='LINKX')
    parser.add_argument('--save_dir', type=str, default='results')

    parser.add_argument('--epochs', type=int, default=200)
    parser.add_argument('--lr', type=float, default=5e-3)
    parser.add_argument('--weight_decay', type=float, default=5e-3)

    parser.add_argument('--dropout', type=float, default=0.5)
    parser.add_argument('--batch_size', type=int, default=1024)
    parser.add_argument('--hidden_dim', type=int, default= 32)

    parser.add_argument('--patience', type=int, default=50)

    parser.add_argument('--ignoreA', action='store_true')
    parser.add_argument('--ignoreX', action='store_true')
    parser.add_argument('--debug', action='store_true')

    parser.add_argument('--sub_dataname', type=str, default='')

    if 'ipykernel' in sys.modules:
        args, _ = parser.parse_known_args()
    else:
        args = parser.parse_args()

    return args

In [8]:
args = get_args()
acc, models = trainer.run_experiment(args, dataset_name='fb100')

Cuda available: True

===== DATASET: fb100 =====
Invalid sub_dataname, deferring to Penn94 graph
Creating Num of classes
Created Num of classes
Model Name: LINKX
Num classes: 2
lr: 0.005 | weight_decay: 0.005 | dropout: 0.5 | batch_size: 1024 | hidden_dim: 32 | patience: 50 | epochs: 200
Using FIXED splits: 5

--- Run 0 ---
Test Accuracy: 0.7562

--- Run 1 ---
Early stopping at epoch 170
Test Accuracy: 0.7500

--- Run 2 ---
Early stopping at epoch 168
Test Accuracy: 0.7460

--- Run 3 ---
Early stopping at epoch 51
Test Accuracy: 0.5327

--- Run 4 ---
Test Accuracy: 0.7464

FB100 → Avg Accuracy: 0.7063 ± 0.0869


In [ ]:
args = get_args()
data = load_data('fb100',args)

In [9]:
for i in range(len(models)):
    torch.save({
        "model_state_dict": models[i].state_dict(),
        "args": vars(args),          # optional but useful
        "num_classes": data.num_classes,
    }, f"linkx_m{i}_fb100.pt")

In [ ]:
from models import LINKXC 

data = load_data('fb100',args)
# embedding = get_emb(models[0], data_)

# loading the m0 model
checkpoint = torch.load("parameters/fb100_penn94/linkx_m0_fb100.pt", map_location=device)
# HeterophilicGNN/LINKX/parameters
model = LINKXC(
    feat_dim=data.num_features,
    hidden_dim=checkpoint["args"]["hidden_dim"],
    num_nodes=data.num_nodes,
    num_classes=data.num_classes,
    ignoreA=checkpoint["args"]["ignoreA"],
    ignoreX=checkpoint["args"]["ignoreX"],
    dropout=checkpoint["args"]["dropout"]
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

Invalid sub_dataname, deferring to Penn94 graph
Creating Num of classes
Created Num of classes


/tmp/ipykernel_23711/1743115251.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("parameters/fb100_penn94/linkx_m0_fb100.pt", map_location=device)

LINKXC(
  (MLPX): Linear(in_features=4814, out_features=32, bias=True)
  (W): Linear(in_features=64, out_features=32, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (out): Linear(in_features=32, out_features=2, bias=True)
)

In [ ]:
embedding = get_emb(model,data)

In [22]:
import torch

def compute_pairwise_distances(emb):
    # emb: [N, D]
    # returns [N, N]
    dist = torch.cdist(emb, emb, p=2)  # Euclidean
    return dist

def build_adj_list(edge_index, num_nodes):
    adj = [set() for _ in range(num_nodes)]
    row, col = edge_index

    for r, c in zip(row.tolist(), col.tolist()):
        adj[r].add(c)
        adj[c].add(r)  # assuming undirected

    return adj

def compute_recall(knn_indices, adj_list):
    recalls = []

    for i in range(len(knn_indices)):
        pred = set(knn_indices[i].tolist())
        true = adj_list[i]

        if len(true) == 0:
            continue

        hit = len(pred & true)
        recall = hit / len(true)

        recalls.append(recall)

    return sum(recalls) / len(recalls)

In [ ]:
def knn_streaming(emb, k=10, chunk_size=4096):
    device = emb.device
    N, D = emb.shape

    knn_indices = []

    for i in range(N):
        x = emb[i:i+1]  # [1, D]

        best_dist = None
        best_idx = None

        for start in range(0, N, chunk_size):
            end = min(start + chunk_size, N)

            chunk = emb[start:end]  # [C, D]

            dist = torch.cdist(x, chunk)  # [1, C]

            if start <= i < end:
                dist[0, i - start] = float('inf')

            if best_dist is None:
                best_dist = dist
                best_idx = torch.arange(start, end, device=device)
            else:
                best_dist = torch.cat([best_dist, dist], dim=1)
                best_idx = torch.cat([best_idx, torch.arange(start, end, device=device)])

            # keep only top-k so memory never grows
            topk = torch.topk(best_dist, k, largest=False)
            best_dist = topk.values
            best_idx = best_idx[topk.indices[0]]

        knn_indices.append(best_idx.cpu())

    return knn_indices

def knn_streaming_per_idx(emb,idx = i,  k=10, chunk_size=4096):
    device = emb.device
    N, D = emb.shape

    knn_indices = []

    for i in range(N):
        x = emb[i:i+1]  # [1, D]

        best_dist = None
        best_idx = None

        for start in range(0, N, chunk_size):
            end = min(start + chunk_size, N)

            chunk = emb[start:end]  # [C, D]

            dist = torch.cdist(x, chunk)  # [1, C]

            if start <= i < end:
                dist[0, i - start] = float('inf')

            if best_dist is None:
                best_dist = dist
                best_idx = torch.arange(start, end, device=device)
            else:
                best_dist = torch.cat([best_dist, dist], dim=1)
                best_idx = torch.cat([best_idx, torch.arange(start, end, device=device)])

            # keep only top-k so memory never grows
            topk = torch.topk(best_dist, k, largest=False)
            best_dist = topk.values
            best_idx = best_idx[topk.indices[0]]

        knn_indices.append(best_idx.cpu())

    return knn_indices

In [24]:
knn_indices = knn_streaming(embedding, k=10, chunk_size=4096)
# knn_indices[0]
adj_list = build_adj_list(data.edge_index, data.num_nodes)
score = compute_recall(knn_indices, adj_list)
print("Neighbor preservation Recall@10:", score)

Neighbor preservation Recall@10: 0.00115046565648372


In [25]:
i = 0
print("Node:", i)
print("True neighbors:", adj_list[i])
print("Embedding neighbors:", knn_indices[i])

Node: 0
True neighbors: {8193, 26632, 521, 38416, 3608, 20001, 3631, 21556, 28212, 1594, 576, 17473, 5200, 12889, 20570, 603, 21596, 26717, 41050, 25695, 21094, 14450, 7289, 20090, 6778, 18043, 24704, 6279, 7818, 4751, 22672, 31897, 7323, 8374, 29371, 12995, 2763, 15565, 1742, 4303, 29908, 38616, 14556, 41189, 18662, 26855, 4843, 34028, 23280, 23793, 26352, 1789, 14595, 23300, 16134, 41224, 13581, 7950, 5903, 32035, 14635, 21292, 37675, 23857, 36658, 26933, 31029, 15167, 36675, 16714, 35149, 6994, 24403, 31572, 29525, 14683, 40796, 24929, 5986, 19811, 32105, 9076, 35200, 29059, 33160, 5513, 22411, 9113, 17311, 27060, 18367, 25023, 18373, 10192, 21970, 17363, 32211, 35794, 40914, 987, 27099, 10210, 34278, 37864, 24554, 22508, 28655, 23549}
Embedding neighbors: tensor([24713,  9189,  9731, 26458, 28894, 33547, 35350, 33419, 30223, 10023])


In [17]:
score = evaluate_embedding_structure(
    embedding,
    data.edge_index,
    k=10
)

print(f"Neighbor Preservation Recall@10: {score:.4f}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 6.43 GiB. GPU 0 has a total capacity of 3.68 GiB of which 1.94 GiB is free. Including non-PyTorch memory, this process has 1.68 GiB memory in use. Of the allocated memory 892.20 MiB is allocated by PyTorch, and 731.80 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)